In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field


@dataclass
class TutorCitation:
    material_id: str
    chunk_id: str
    page_number: int | None = None
    file_name: str | None = None


@dataclass
class TutorAIResponse:
    answer: str
    citations: list[TutorCitation] = field(default_factory=list)
    grounded: bool = False


class TutorAI:
    def __init__(self, ai_service):
        self.ai_service = ai_service

    @staticmethod
    def _approx_tokens(text: str) -> int:
        return (len(text) + 3) // 4

    @staticmethod
    def _trim_text(text: str, limit: int) -> str:
        text = str(text or "")
        return text if len(text) <= limit else text[:limit].rstrip() + "…"

    def answer(
        self,
        question: str,
        evidence_chunks: list,
        conversation: str = "",
        learning_context: str = "",
        user_id: str | None = None,
        project_id: str | None = None,
    ) -> TutorAIResponse:
        from app.ai.prompts import TUTOR_SYSTEM_PROMPT, build_tutor_prompt

        question = question.strip()
        if not question:
            raise ValueError("Tutor question cannot be empty.")

        if not evidence_chunks:
            return TutorAIResponse(
                answer="I don't have enough information in the available study material to answer that reliably.",
                citations=[], grounded=False,
            )

        citations = []
        usable_chunks = []
        for chunk in evidence_chunks:
            chunk_text = str(getattr(chunk, "text", "") or "").strip()
            if not chunk_text:
                continue
            chunk_id = getattr(chunk, "chunk_id", "")
            material_id = getattr(chunk, "material_id", "")
            page_number = getattr(chunk, "page_number", None)
            file_name = getattr(chunk, "source_file_name", None)
            usable_chunks.append((chunk, chunk_id, material_id, page_number, file_name))
            citations.append(TutorCitation(material_id=material_id, chunk_id=chunk_id, page_number=page_number, file_name=file_name))

        if not usable_chunks:
            return TutorAIResponse(
                answer="I don't have enough usable information in the available study material to answer that reliably.",
                citations=[], grounded=False,
            )

        # Keep the current question intact. Reduce evidence/context before the model call.
        evidence_limits = [1200, 900, 700, 500, 350]
        history_limits = [3500, 2500, 1800, 1200, 700]
        learning_limits = [1800, 1200, 900, 600, 400]
        max_prompt_tokens = 5000

        prompt = ""
        for evidence_limit, history_limit, learning_limit in zip(evidence_limits, history_limits, learning_limits):
            evidence_parts = []
            for chunk, chunk_id, _, page_number, _ in usable_chunks[:5]:
                text = self._trim_text(getattr(chunk, "text", ""), evidence_limit)
                evidence_parts.append(f"[Chunk {chunk_id} | Page {page_number}]\n{text}")
            evidence = "\n\n".join(evidence_parts)
            prompt = build_tutor_prompt(
                question=question,
                evidence=evidence,
                conversation=self._trim_text(conversation, history_limit),
                learning_context=self._trim_text(learning_context, learning_limit),
            )
            if self._approx_tokens(prompt) <= max_prompt_tokens:
                break

        if self._approx_tokens(prompt) > max_prompt_tokens:
            # Final fallback keeps the question and citation metadata while minimizing evidence/context.
            evidence_parts = []
            for chunk, chunk_id, _, page_number, _ in usable_chunks[:3]:
                text = self._trim_text(getattr(chunk, "text", ""), 250)
                evidence_parts.append(f"[Chunk {chunk_id} | Page {page_number}]\n{text}")
            prompt = build_tutor_prompt(
                question=question,
                evidence="\n\n".join(evidence_parts),
                conversation=self._trim_text(conversation, 300),
                learning_context=self._trim_text(learning_context, 250),
            )

        if self._approx_tokens(prompt) > max_prompt_tokens:
            raise ValueError("Tutor prompt exceeds the safe token budget after trimming.")

        print(
            "[TutorAI] "
            f"evidence_chunks={len(usable_chunks[:5])} "
            f"approx_prompt_tokens={self._approx_tokens(prompt)} "
            f"question_length={len(question)}"
        )

        answer = self.ai_service.generate_text(
            prompt=prompt,
            system_instruction=TUTOR_SYSTEM_PROMPT,
            operation="tutor",
            user_id=user_id,
            project_id=project_id,
        )

        answer = answer.strip()
        if not answer:
            raise RuntimeError("Tutor model returned an empty answer.")

        return TutorAIResponse(answer=answer, citations=citations, grounded=True)
